# PatchTST — Walmart Store Sales Forecasting

MLflow experiment: `PatchTST_Training`.
Runs: `PatchTST_DataPrep`, `PatchTST_CrossValidation`, `PatchTST_HPO`, `PatchTST_Final`.

Global transformer over patched series via Nixtla's `neuralforecast`, trained with the
exogenous features from the shared `features.build_features`. Metric: WMAE, holiday weeks
weighted 5x. Validation: expanding window, horizon = 39 weeks (same folds as the tree models).

There is no separate `nf_prep.py` in this repo yet (see `TASKS.md` T13), so the Nixtla
long-format conversion is done inline in this notebook (`to_nixtla` / `from_nixtla` below).
If `nf_prep.py` lands later, swap these two functions for the shared import.

## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    os.chdir("/content")          # re-running this cell must not nest the clone
    if not os.path.isdir("/content/ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd /content/ML-final

    !pip -q install mlflow dagshub kaggle neuralforecast optuna pandas pyarrow

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("/content/ML-final")

print("IN_COLAB =", IN_COLAB)


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
print("IN_COLAB =", IN_COLAB, "| ROOT =", ROOT)


## Robust data directory (Fix 1)

In [ ]:
from pathlib import Path

def find_data_dir(root):
    """Locate the folder that actually holds the 5 Kaggle CSVs.

    Handles both layouts: a flat data/ folder, or the nested folder Kaggle's
    zip produces (data/walmart-recruiting-store-sales-forecasting/).
    """
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("DATA_DIR =", DATA_DIR)


## MLflow / DagsHub tracking (Fix 2)

In [ ]:
import os
import mlflow

for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

if IN_COLAB:
    from google.colab import userdata
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(ROOT / ".env")
    except ImportError:
        pass

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/karev23/ML-final.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"

EXPERIMENT_NAME = "PatchTST_Training"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
print("tracking:", mlflow.get_tracking_uri())

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)


## Imports and data

In [ ]:
import numpy as np
import pandas as pd

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import PatchTST
    from neuralforecast.losses.pytorch import MAE
except ImportError as e:
    raise ImportError(
        "neuralforecast is required for this notebook. Install requirements-dl.txt "
        "in its own environment (see README) before running this cell."
    ) from e

train, test, features, stores, sample = load_raw(DATA_DIR)
train_merged = merge_all(train, features, stores).reset_index(drop=True)
raw_test = test[["Store", "Dept", "Date", "IsHoliday"]].reset_index(drop=True)
dates = train["Date"].to_numpy()
print("train_merged:", train_merged.shape, "| raw_test:", raw_test.shape)

## Run: PatchTST_DataPrep

Convert to Nixtla long format (`unique_id`, `ds`, `y` + numeric exogenous columns).

Two constraints shape this notebook and both differ from the tree models:

**Validation is a single 39-week holdout, not the 3-fold expanding window.** PatchTST needs
`input_size + h` weeks of history per series to form one training window. With `h = 39` and only
143 weeks of data in total, the expanding-window folds leave 26 / 65 / 104 weeks of training
history - not enough for a 39-week-horizon sequence model to build a single window in any fold.
`validation.holdout_split` leaves 104 training weeks, which supports `input_size` up to 65.

**Exogenous features exclude everything derived from `Weekly_Sales`.** The lag/history columns
(`sales_lag_52`, `*_history_mean`) are unknown over a future horizon, so they cannot be passed as
`futr_exog`. PatchTST learns the sales dynamics from `y` itself; the exogenous channel carries only
covariates genuinely known in advance (calendar, markdowns, store, temperature, fuel, CPI).

In [ ]:
SALES_DERIVED = ("sales_lag", "history_mean")

def _exog_matrix(df_merged):
    """Numeric covariates known in advance. Sales-derived columns are dropped: they
    do not exist over a future horizon and would leak the target in validation."""
    src = df_merged.drop(columns=["Weekly_Sales"], errors="ignore")
    X = build_features(src, sales_history_df=None, encode_categoricals=True)
    X = X.select_dtypes(include=[np.number])
    keep = [c for c in X.columns if not any(k in c for k in SALES_DERIVED)]
    return X[keep].astype("float32")

def to_nixtla(df_merged, fit_columns=None):
    X = _exog_matrix(df_merged)
    if fit_columns is not None:
        X = X.reindex(columns=fit_columns, fill_value=0)
    nf = pd.DataFrame({
        "unique_id": df_merged["Store"].astype(str) + "_" + df_merged["Dept"].astype(str),
        "ds": pd.to_datetime(df_merged["Date"]),
    })
    if "Weekly_Sales" in df_merged.columns:
        nf["y"] = df_merged["Weekly_Sales"].to_numpy(dtype=float)
    for col in X.columns:
        nf[col] = X[col].to_numpy()
    return nf, list(X.columns)

H = 39
INPUT_SIZE = 52          # 52 + 39 = 91 <= 104 training weeks in the holdout
MIN_HISTORY = INPUT_SIZE + H

nf_train_full, EXOG_COLS = to_nixtla(train_merged)
series_counts = nf_train_full.groupby("unique_id")["ds"].count()
short_series = series_counts[series_counts < MIN_HISTORY].index

with mlflow.start_run(run_name="PatchTST_DataPrep"):
    safe_log_param("horizon", H)
    safe_log_param("input_size", INPUT_SIZE)
    safe_log_param("min_history_weeks", MIN_HISTORY)
    safe_log_param("n_exogenous_cols", len(EXOG_COLS))
    safe_log_param("n_series_total", int(series_counts.shape[0]))
    safe_log_param("n_series_dropped_short", int(len(short_series)))
    safe_log_metric("nf_train_rows", float(len(nf_train_full)))
    print(f"series: {series_counts.shape[0]} total, {len(short_series)} too short (<{MIN_HISTORY}w)")
    print("exogenous columns:", len(EXOG_COLS))

## Run: PatchTST_CrossValidation

Single 39-week holdout (see above for why the expanding window is unusable here). Compares
**with vs without exogenous** features; the winner sets `USE_EXOG` for HPO and the final fit.

In [ ]:
from validation import holdout_split

TR_MASK, VA_MASK = holdout_split(dates, horizon=H)

def run_patchtst(tr_mask, va_mask, use_exog, model_kwargs):
    tr = train_merged[tr_mask].reset_index(drop=True)
    va = train_merged[va_mask].reset_index(drop=True)

    nf_tr, fit_cols = to_nixtla(tr)
    counts = nf_tr.groupby("unique_id")["ds"].count()
    need = model_kwargs["input_size"] + H
    keep_ids = counts[counts >= need].index
    nf_tr = nf_tr[nf_tr["unique_id"].isin(keep_ids)].reset_index(drop=True)

    exog = fit_cols if use_exog else []
    model = PatchTST(h=H, loss=MAE(), futr_exog_list=exog, hist_exog_list=exog, **model_kwargs)
    nf = NeuralForecast(models=[model], freq="W-FRI")
    nf.fit(df=nf_tr[["unique_id", "ds", "y"] + exog])

    nf_va, _ = to_nixtla(va, fit_columns=fit_cols)
    futr_df = nf_va[["unique_id", "ds"] + exog]
    preds = nf.predict(futr_df=futr_df).reset_index()

    truth = va.assign(unique_id=va["Store"].astype(str) + "_" + va["Dept"].astype(str),
                      ds=pd.to_datetime(va["Date"]))
    eva = truth[["unique_id", "ds", "Weekly_Sales", "IsHoliday"]].merge(
        preds[["unique_id", "ds", "PatchTST"]], on=["unique_id", "ds"], how="left"
    )
    fallback = float(nf_tr["y"].mean())  # series too short to model
    p = np.clip(eva["PatchTST"].fillna(fallback).to_numpy(), 0, None)
    return wmae(eva["Weekly_Sales"].to_numpy(float), p, eva["IsHoliday"].astype(bool).to_numpy())

BASE_MODEL_KWARGS = dict(input_size=INPUT_SIZE, patch_len=16, stride=8, hidden_size=128,
                         n_heads=4, max_steps=200, val_check_steps=50, random_seed=42)

with mlflow.start_run(run_name="PatchTST_CrossValidation"):
    wmae_exog = run_patchtst(TR_MASK, VA_MASK, True, BASE_MODEL_KWARGS)
    wmae_noexog = run_patchtst(TR_MASK, VA_MASK, False, BASE_MODEL_KWARGS)
    safe_log_params(BASE_MODEL_KWARGS)
    safe_log_param("validation", "holdout_39w")
    safe_log_metric("wmae_holdout_exog", wmae_exog)
    safe_log_metric("wmae_holdout_noexog", wmae_noexog)
    print(f"holdout WMAE  exog={wmae_exog:,.1f}   no-exog={wmae_noexog:,.1f}")

USE_EXOG = wmae_exog <= wmae_noexog
print("USE_EXOG =", USE_EXOG)

## Run: PatchTST_HPO

Optuna over the main PatchTST hyperparameters, each trial a nested run. `input_size` is capped at
65 because the holdout leaves 104 training weeks and a window costs `input_size + 39`.

In [ ]:
import optuna

N_TRIALS = 5

def objective(trial):
    kwargs = dict(
        input_size=trial.suggest_categorical("input_size", [26, 39, 52, 65]),
        patch_len=trial.suggest_categorical("patch_len", [8, 16, 24]),
        stride=trial.suggest_categorical("stride", [4, 8]),
        hidden_size=trial.suggest_categorical("hidden_size", [64, 128, 256]),
        n_heads=trial.suggest_categorical("n_heads", [2, 4, 8]),
        max_steps=200,
        val_check_steps=50,
        random_seed=42,
    )
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        score = run_patchtst(TR_MASK, VA_MASK, USE_EXOG, kwargs)
        safe_log_params(kwargs)
        safe_log_metric("wmae_holdout", score)
    return score

with mlflow.start_run(run_name="PatchTST_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    safe_log_params(study.best_params)
    safe_log_metric("wmae_holdout_best", study.best_value)
    print("best WMAE:", round(study.best_value, 1))
    print("best params:", study.best_params)

BEST_MODEL_KWARGS = dict(BASE_MODEL_KWARGS, **study.best_params, max_steps=300)
BEST_WMAE = float(study.best_value)

## Run: PatchTST_Final

Refit on the full 143 weeks and log a raw->prediction `pyfunc`
(`neuralforecast_pyfunc.NeuralForecastRawPyFunc`), matching the Pipeline contract the tree models
use, then register it. This is what `model_inference.ipynb` loads.

In [ ]:
import mlflow.pyfunc
from neuralforecast_pyfunc import NeuralForecastRawPyFunc

nf_final_df, FINAL_EXOG = to_nixtla(train_merged)
need = BEST_MODEL_KWARGS["input_size"] + H
counts = nf_final_df.groupby("unique_id")["ds"].count()
nf_final_df = nf_final_df[nf_final_df["unique_id"].isin(counts[counts >= need].index)].reset_index(drop=True)

exog = FINAL_EXOG if USE_EXOG else []
final_model = PatchTST(h=H, loss=MAE(), futr_exog_list=exog, hist_exog_list=exog, **BEST_MODEL_KWARGS)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=nf_final_df[["unique_id", "ds", "y"] + exog])

pyfunc_model = NeuralForecastRawPyFunc(
    neuralforecast_model=nf_final,
    features_df=features,
    stores_df=stores,
    train_history_raw=train,
    model_alias="PatchTST",
    uses_exog=USE_EXOG,
    exog_cols=exog,
)

test_pred = pyfunc_model.predict(None, raw_test)
print("predicted test rows:", test_pred.shape, "| sample:", np.round(test_pred[:5], 1))

make_submission(raw_test, test_pred, "submission_patchtst.csv")
print("wrote submission_patchtst.csv")

with mlflow.start_run(run_name="PatchTST_Final"):
    safe_log_params(BEST_MODEL_KWARGS)
    safe_log_param("use_exog", USE_EXOG)
    safe_log_param("validation", "holdout_39w")
    safe_log_metric("wmae_holdout", BEST_WMAE)
    mlflow.pyfunc.log_model(
        name="model",
        python_model=pyfunc_model,
        code_paths=["neuralforecast_pyfunc.py", "features.py", "dataloader.py"],
        input_example=raw_test.head(3),
        registered_model_name="PatchTST_Walmart",
    )
    print("logged + registered pyfunc | holdout WMAE:", round(BEST_WMAE, 1))

## Results

- CV WMAE: ___  | Kaggle public LB WMAE: ___
- Exogenous features helped: ___
- Best HPO config: ___
- vs LightGBM / XGBoost: ___
- vs DLinear / N-BEATS: ___
